* input : noondata, year selection, vessels/fleet selection
* vessel type differences are not included. ie, TEU columns only need to be filled when the vessel is a container

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
year = 2023
vessels = [9004231, 9369734, 9294408, 9193680, 9294824, 9241451, 9770749, 9770751, 9403396, 9241463, 9241475, 9401776, 9456965, 9463281, 9231262, 9231250, 9288409]

In [3]:
noon = pd.read_csv('fleet6.csv')

noon = noon[noon['imo'].isin(vessels)]
data = noon[['imo','vessel','report_date_time','voyage_order','cargo_total','cargo_total_teu','miles_by_gps','manvrng_miles_by_gps',
            'fuel_me_rsdl_vls', 'fuel_me_rsdl_hs', 'fuel_me_rsdl_uls', 'fuel_me_dstlt_vls', 'fuel_me_dstlt_uls', 'fuel_me_tnktnr_dstlt_vls',
            'fuel_aux_rsdl_vls', 'fuel_aux_rsdl_hs', 'fuel_aux_rsdl_uls', 'fuel_aux_dstlt_vls', 'fuel_aux_dstlt_uls', 'fuel_aux_tnktnr_dstlt_vls',
             'fuel_boiler_rsdl_vls', 'fuel_boiler_rsdl_hs', 'fuel_boiler_rsdl_uls', 'fuel_boiler_dstlt_vls', 'fuel_boiler_dstlt_uls', 'fuel_boiler_tnktnr_dstlt_vls',
            'fuel_me_hs', 'fuel_aux_hs', 'fuel_boiler_hs', 'fuel_me_ls', 'fuel_aux_ls', 'fuel_boiler_ls',
            'fuel_me_mgo', 'fuel_me_mgo_ls', 'fuel_me_mdo', 'fuel_aux_mgo', 'fuel_aux_mgo_ls', 'fuel_aux_mdo', 'fuel_boiler_mgo', 'fuel_boiler_mgo_ls', 'fuel_boiler_mdo']]

data['report_date_time'] = pd.to_datetime(data['report_date_time'])
data = data.sort_values(by='report_date_time')
data['year'] = data['report_date_time'].dt.year

data = data[data['year']==year]

In [4]:
columns_to_fill = [
    'cargo_total','cargo_total_teu','miles_by_gps','manvrng_miles_by_gps',
    'fuel_me_rsdl_vls', 'fuel_me_rsdl_hs', 'fuel_me_rsdl_uls', 'fuel_me_dstlt_vls', 'fuel_me_dstlt_uls', 'fuel_me_tnktnr_dstlt_vls',
    'fuel_aux_rsdl_vls', 'fuel_aux_rsdl_hs', 'fuel_aux_rsdl_uls', 'fuel_aux_dstlt_vls', 'fuel_aux_dstlt_uls', 'fuel_aux_tnktnr_dstlt_vls',
    'fuel_boiler_rsdl_vls', 'fuel_boiler_rsdl_hs', 'fuel_boiler_rsdl_uls', 'fuel_boiler_dstlt_vls', 'fuel_boiler_dstlt_uls', 'fuel_boiler_tnktnr_dstlt_vls',
    'fuel_me_hs', 'fuel_aux_hs', 'fuel_boiler_hs', 'fuel_me_ls', 'fuel_aux_ls', 'fuel_boiler_ls',
    'fuel_me_mgo', 'fuel_me_mgo_ls', 'fuel_me_mdo', 'fuel_aux_mgo', 'fuel_aux_mgo_ls', 'fuel_aux_mdo', 'fuel_boiler_mgo', 'fuel_boiler_mgo_ls', 'fuel_boiler_mdo'
]

data[columns_to_fill] = data[columns_to_fill].fillna(0)

In [7]:


data['distance'] = np.where(data['miles_by_gps'] != data['manvrng_miles_by_gps'],
                data['miles_by_gps'] + data['manvrng_miles_by_gps'],
                data['miles_by_gps'])

data['hfo'] = np.where(data['year'] >=2020,
                data[['fuel_me_rsdl_hs', 'fuel_aux_rsdl_hs', 'fuel_boiler_rsdl_hs']].sum(axis=1),
                data[['fuel_me_hs', 'fuel_aux_hs', 'fuel_boiler_hs']].sum(axis=1))

data['lfo'] = np.where(data['year'] >=2020,
                data[["fuel_me_rsdl_vls", "fuel_me_rsdl_uls", "fuel_aux_rsdl_vls", "fuel_aux_rsdl_uls", "fuel_boiler_rsdl_vls", "fuel_boiler_rsdl_uls"]].sum(axis=1),
                data[['fuel_me_ls', 'fuel_aux_ls', 'fuel_boiler_ls']].sum(axis=1))
data['mgo'] = np.where(data['year'] >=2020,
                data[["fuel_me_dstlt_vls", "fuel_me_dstlt_uls", "fuel_me_tnktnr_dstlt_vls", "fuel_aux_dstlt_vls", "fuel_aux_dstlt_uls", "fuel_aux_tnktnr_dstlt_vls", "fuel_boiler_dstlt_vls", "fuel_boiler_dstlt_uls", "fuel_boiler_tnktnr_dstlt_vls"]].sum(axis=1),
                data[['fuel_me_mgo', 'fuel_me_mgo_ls', 'fuel_me_mdo', 'fuel_aux_mgo', 'fuel_aux_mgo_ls', 'fuel_aux_mdo', 'fuel_boiler_mgo', 'fuel_boiler_mgo_ls', 'fuel_boiler_mdo']].sum(axis=1))

data['total_fuel'] = data[['hfo', 'lfo', 'mgo']].sum(axis=1)

data['CO2'] = (data['hfo']*3.114 + data['lfo']*3.151 + data['mgo']*3.206)

In [8]:
data_voy = data.groupby(['imo', 'voyage_order']).agg({
    'distance' : 'sum',
    'hfo': 'sum',
    'lfo': 'sum',
    'mgo': 'sum',
    'total_fuel' : 'sum',
    'CO2': 'sum',
    'cargo_total': 'first',
    'cargo_total_teu': 'first'
}).reset_index()

data_voy['transport_work_mt'] = data_voy['cargo_total']*data_voy['distance']
data_voy['transport_work_teu'] = data_voy['cargo_total_teu']*data_voy['distance']

In [9]:
data_imo = data_voy.groupby(['imo']).agg({
    'voyage_order' : 'count',
    'distance' : 'sum',
    'hfo': 'sum',
    'lfo': 'sum',
    'mgo': 'sum',
    'total_fuel' : 'sum',
    'CO2': 'sum',
    'cargo_total': 'sum',
    'cargo_total_teu': 'sum',
    'transport_work_mt': 'sum',
    'transport_work_teu': 'sum'
}).reset_index()
data_imo.rename(columns={'voyage_order': 'number_of_voyages'}, inplace=True)

data_imo['eeoi_mt'] = data_imo['CO2']*10**6/data_imo['transport_work_mt']
data_imo['eeoi_teu'] = data_imo['CO2']*10**6/data_imo['transport_work_teu']
data_imo['cargo_mt_per_voyage'] = data_imo['cargo_total']/data_imo['number_of_voyages']
data_imo['cargo_teu_per_voyage'] = data_imo['cargo_total_teu']/data_imo['number_of_voyages']
data_imo['distance_per_voyage'] = data_imo['distance']/data_imo['number_of_voyages']
data_imo['consumption_per_voyage'] = data_imo['total_fuel']/data_imo['number_of_voyages']
data_imo['emission_per_voyage'] = data_imo['CO2']/data_imo['number_of_voyages']


In [10]:
data_imo

,imo,number_of_voyages,distance,hfo,lfo,mgo,total_fuel,CO2,cargo_total,cargo_total_teu,transport_work_mt,transport_work_teu,eeoi_mt,eeoi_teu,cargo_mt_per_voyage,cargo_teu_per_voyage,distance_per_voyage,consumption_per_voyage,emission_per_voyage
0,9004231,105,43128.08000,0.000,1719.464,1033.478,2752.942,8731.361532,499352.40,43769.0,2.098511e+08,1.837014e+07,41.607422,475.301978,4755.737143,416.847619,410.743619,26.218495,83.155824
1,9193680,67,43546.39999,4281.417,76.070,429.285,4786.772,14948.316818,922993.91,63216.0,6.509515e+08,3.859737e+07,22.963795,387.288528,13776.028507,943.522388,649.946269,71.444358,223.109206
2,9231250,73,68008.10000,12329.602,0.000,1549.037,13878.639,43360.593250,2326101.00,189021.0,2.607471e+09,2.233459e+08,16.629368,194.140996,31864.397260,2589.328767,931.617808,190.118342,593.980729
3,9231262,32,54207.40000,7905.630,74.630,2065.146,10045.406,31474.149026,1017938.80,84523.0,1.783236e+09,1.699625e+08,17.650021,185.182884,31810.587500,2641.343750,1693.981250,313.918938,983.567157
4,9241451,32,26334.00000,0.000,3048.375,64.800,3113.175,9813.178425,419016.00,35088.0,3.390217e+08,3.077367e+07,28.945577,318.882291,13094.250000,1096.500000,822.937500,97.286719,306.661826
5,9241463,54,46494.13000,0.000,5349.900,271.560,5621.460,17728.156260,772679.20,54393.0,7.471807e+08,4.953857e+07,23.726732,357.865751,14308.874074,1007.277778,861.002407,104.101111,328.299190
6,9241475,51,51704.90000,0.000,5660.765,127.450,5788.215,18245.675215,370339.00,55481.0,4.637242e+08,6.691984e+07,39.345960,272.649722,7261.549020,1087.862745,1013.821569,113.494412,357.758338
7,9288409,42,52412.20000,0.000,7772.170,2281.290,10053.460,31803.923410,954003.40,88441.0,1.328265e+09,1.409122e+08,23.943952,225.700338,22714.366667,2105.738095,1247.909524,239.368095,757.236272
8,9294408,38,29737.50000,0.000,3666.850,2292.610,5959.460,18904.352010,537578.70,54851.0,5.132131e+08,4.954495e+07,36.835286,381.559610,14146.807895,1443.447368,782.565789,156.827895,497.482948
9,9294824,54,51069.30000,0.000,8570.338,544.585,9114.923,28751.074548,1231038.10,116269.0,1.378966e+09,1.293337e+08,20.849735,222.301473,22797.001852,2153.129630,945.727778,168.794870,532.427306
